# Alaska salmon data cleaning

This notebook holds the code to clean the Alaskan salmon dataset found on [KNB](https://knb.ecoinformatics.org/view/doi:10.5063/F1707ZTM). This data set spans 1922 to 2017 and contains salmon measurements, sample locations, and type of salmon capture. The final result is a local relational database that makes this data more usable. It can be recreated by following the steps from the Alaska salmon relational database repository. 

In [ ]:
# import necessary libraries
import pandas as pd
import numpy as np
import os
import janitor

In [2]:
# load in data
salmon_data = pd.read_csv("Alaskan-salmon1922-2017/data/ASL_master.csv")
length_type = pd.read_csv("Alaskan-salmon1922-2017/data/length_type_lookup.csv")
location = pd.read_csv("Alaskan-salmon1922-2017/data/Locations_subdistricts_uniqueID.csv")
project_type = pd.read_csv("Alaskan-salmon1922-2017/data/ASLProjectType.csv")
gear =  pd.read_csv("Alaskan-salmon1922-2017/data/gear.csv")


/var/folders/kj/1ybgv25d7zd06v9ndqdccd2r0000gn/T/ipykernel_24973/1368001829.py:2: DtypeWarning: Columns (1,3,8,11,12,13,15,16,17,18,26) have mixed types. Specify dtype option on import or set low_memory=False.
  salmon_data = pd.read_csv("Alaskan-salmon1922-2017/data/ASL_master.csv")


## Salmon data
Preparing the salmon data to be entered into the relational database. 

In [3]:
# assess whether the data types are what we expect
salmon_data.dtypes

Species                      object
Length.Measurement.Type      object
sampleYear                  float64
ASLProjectType               object
LocationID                   object
sampleDate                   object
Length                      float64
Weight                      float64
Sex                          object
Salt.Water.Age              float64
DataSource                   object
cardNo                       object
fishNum                      object
Age.Error                    object
Fresh.Water.Age             float64
Sex.Determination.Method     object
subSystem                    object
Flag                         object
Gear                         object
SASAP.Region                 object
LocationUnique               object
DistrictID                  float64
Sub.DistrictID              float64
Stat.area                   float64
Lat                         float64
Lon                         float64
AWC_CODE                     object
dtype: object

Yay! It looks like all of the data is in the right data type except for the sampleYear and sampleDate. But, there is no primary key. 
 
### Make a primary key for our salmon dataframe
A database requires a primary key that has no NA values and is unique to each row. Add an ID column to the salmon data to fullfil this requirement. 

In [4]:
# add an id column to the salmon dataframe to be used as a primary key
salmon_data['salmon_id'] = salmon_data.index

In [5]:
# initial look at the dataframe
salmon_data.head()

,Species,Length.Measurement.Type,sampleYear,ASLProjectType,LocationID,sampleDate,Length,Weight,Sex,Salt.Water.Age,...,Gear,SASAP.Region,LocationUnique,DistrictID,Sub.DistrictID,Stat.area,Lat,Lon,AWC_CODE,salmon_id
0,chinook,length not taken,1992.0,commercial catch,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,3.0,...,troll,Southeast,Affleck Canal/Spanish Is/Louise Cove-commercia...,105.0,10.0,10510.0,NaN,NaN,NaN,0
1,chinook,length not taken,1992.0,commercial catch,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,3.0,...,troll,Southeast,Affleck Canal/Spanish Is/Louise Cove-commercia...,105.0,10.0,10510.0,NaN,NaN,NaN,1
2,chinook,length not taken,1992.0,commercial catch,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,3.0,...,troll,Southeast,Affleck Canal/Spanish Is/Louise Cove-commercia...,105.0,10.0,10510.0,NaN,NaN,NaN,2
3,chinook,length not taken,1992.0,commercial catch,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,3.0,...,troll,Southeast,Affleck Canal/Spanish Is/Louise Cove-commercia...,105.0,10.0,10510.0,NaN,NaN,NaN,3
4,chinook,length not taken,1992.0,commercial catch,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,4.0,...,troll,Southeast,Affleck Canal/Spanish Is/Louise Cove-commercia...,105.0,10.0,10510.0,NaN,NaN,NaN,4


### Adjust data type 

In [36]:
# convert salmon year to an int so we can give it type date in the database
salmon_data['sampleYear'] = salmon_data['sampleYear'].astype('Int64')

### Subset for the necessary data 

We don't need all of the columns for this analysis so we can subset. 

In [6]:
# select the variables that are essential for the salmon dataframe 
salmon_data = salmon_data[['Species', 'Length.Measurement.Type', 'sampleYear', 'LocationID', 'sampleDate', 'Length', 'Weight', 'Sex','Salt.Water.Age', 'Fresh.Water.Age', 'Gear', 'Stat.area', 'DataSource']]

# Length dataset

In [ ]:
# view the length dataset
length_type.head()

,Length.Measurement.Type,Length_fill,LocationID,Species,ASLProjectType,sampleYear
0,mid-eye to fork of tail,NaN,108 Creek,coho,escapement,1998.0
1,mid-eye to fork of tail,NaN,108 Creek,coho,escapement,1999.0
2,mid-eye to fork of tail,NaN,108 Creek,coho,escapement,2000.0
3,mid-eye to fork of tail,NaN,18 Mile Slough,chum,escapement,2003.0
4,mid-eye to fork of tail,NaN,18 Mile Slough,chum,escapement,2004.0


In [9]:
# assess if the data types for the length need to be adjusted
length_type.dtypes

Length.Measurement.Type     object
Length_fill                 object
LocationID                  object
Species                     object
ASLProjectType              object
sampleYear                 float64
dtype: object

All of the data seems to be in adequate formats but above we just adjusted the year column to be a type integer. We should change the datatype of this year column to match. 

### Adjust datatype

In [ ]:
# check all of the unique values of the year column 
length_type['sampleYear'].unique()

array([1998., 1999., 2000., 2003., 2004., 1984., 1983., 1985., 1986.,
       1987., 1988., 1990., 1991., 1992., 1993., 2002., 2005., 2006.,
       2007., 2008., 1989., 1996., 2010., 2011., 2013., 2014., 1994.,
       1995., 1997., 2001., 2009., 2012., 2015., 1982., 1965., 1967.,
       1964., 1970., 1971., 1972., 1974., 1975., 1963., 1966., 1968.,
       1969., 1973.,   nan, 1976., 1960., 1978., 2016., 1979., 1980.,
       1981., 1977., 1961., 2017., 1962., 1959., 1958., 1957.])

In [11]:
# change year from float ot int
length_type['sampleYear'] = length_type['sampleYear'].astype('Int64')

# Location dataset

In [10]:
# assess if all of the data is in the right format
location.dtypes

SASAP.Region               object
SASAP.Region_Corrected     object
Location                   object
ASLProjectType             object
District                   object
Sub.District               object
LocationUnique             object
DistrictID                float64
Sub.DistrictID            float64
Stat.area                 float64
Lat                       float64
Lon                       float64
AWC_CODE                   object
source                     object
LocationID                 object
dtype: object

In [ ]:
# view the location data
location.head()

,SASAP.Region,SASAP.Region_Corrected,Location,ASLProjectType,District,Sub.District,LocationUnique,DistrictID,Sub.DistrictID,Stat.area,Lat,Lon,AWC_CODE,source,LocationID
0,Alaska Peninsula and Aleutian Islands,Alaska Peninsula and Aleutian Islands,NaN,commercial catch,302,19,Aleutians-commercial catch-302,302.0,NaN,NaN,NaN,NaN,NaN,NaN,Aleutians
1,Alaska Peninsula and Aleutian Islands,Alaska Peninsula and Aleutian Islands,Bear River,commercial catch,315,11,Bear River-commercial catch-31511,315.0,11.0,31511.0,NaN,NaN,NaN,NaN,Bear River
2,Alaska Peninsula and Aleutian Islands,Alaska Peninsula and Aleutian Islands,Bear River,escapement,315,11,Bear River-escapement-31511,315.0,11.0,31511.0,56.0389,-160.2734,315-11-10200,ADFG,Bear River
3,Alaska Peninsula and Aleutian Islands,Alaska Peninsula and Aleutian Islands,Bear River,test fishing,315,11,Bear River-test fishing-31511,315.0,11.0,31511.0,56.0389,-160.2734,315-11-10200,ADFG,Bear River
4,Alaska Peninsula and Aleutian Islands,Alaska Peninsula and Aleutian Islands,Bear River,test fishing,316,11,Bear River-test fishing-31511,315.0,11.0,31511.0,56.0389,-160.2734,315-11-10200,ADFG,Bear River


In [15]:
# subset for only necessary rows
location = location[['LocationUnique', 'SASAP.Region_Corrected','Location', 'ASLProjectType', 'District', 'Sub.District', 'Lat', 'Lon', 'LocationID']]

In [16]:
# change the name of the columns for easier interpretation
location_rename = location.rename(columns = {'SASAP.Region_Corrected' : 'LocationID'})

In [17]:
# adjust the location data
location = location.clean_names()

### Location data key

The primary key of the dataframe must be unique and non-null. Let's ensure it meets all of these qualifications.

In [18]:
# view duplicates based on the location unique
dup_location = location[location.duplicated(subset = ['locationunique'], keep = False)]
dup_location

,locationunique,sasap_region_corrected,location,aslprojecttype,district,sub_district,lat,lon,locationid
1,Bear River-commercial catch-31511,Alaska Peninsula and Aleutian Islands,Bear River,commercial catch,315,11,NaN,NaN,Bear River
2,Bear River-escapement-31511,Alaska Peninsula and Aleutian Islands,Bear River,escapement,315,11,56.038900,-160.273400,Bear River
3,Bear River-test fishing-31511,Alaska Peninsula and Aleutian Islands,Bear River,test fishing,315,11,56.038900,-160.273400,Bear River
4,Bear River-test fishing-31511,Alaska Peninsula and Aleutian Islands,Bear River,test fishing,316,11,56.038900,-160.273400,Bear River
5,Bear River-escapement-31511,Alaska Peninsula and Aleutian Islands,Bear River,escapement,315,12,56.038900,-160.273400,Bear River
...,...,...,...,...,...,...,...,...,...
2735,Rampart Rapids-subsistence catch-33452,Yukon,Rampart Rapids,subsistence catch,Yukon,Y5 (Subdistrict 5),65.337055,-151.082611,Rampart Rapids
2736,Rampart Rapids-commercial catch-33452,Yukon,Rapids Research Center,commercial catch,Yukon,Y5 (Subdistrict 5),NaN,NaN,Rampart Rapids
2737,Rampart Rapids-subsistence catch-33452,Yukon,Rapids Research Center,subsistence catch,Yukon,Y5 (Subdistrict 5),65.337055,-151.082611,Rampart Rapids
2763,Tanana River-escapement-33460,Yukon,Tanana River,escapement,Yukon,Y6 (Subdistrict 6),64.979281,-150.845454,Tanana River


The location unique does in fact have duplicates as shown in the subset above. The column `locationunique` column appears to be created by joining the `location`, `aslprojecttype`, `district`, and `subdistrict` columns. These duplicates seem to just be errors in the creation of the key. We can remake the `locationunique` to avoid any issues when ingesting the data into the relational database.

Additionally, some of the rows needed to recreate the `locationunique` column are NA. To avoid losing any data we can fill the NA with the well known NA value of `999`. This will make unique `locationunique`.

In [19]:
# fill the na in sub_district with a filler 999
location['sub_district'] = location['sub_district'].fillna('999')
location['district'] = location['district'].fillna('999')
location['aslprojecttype'] = location['aslprojecttype'].fillna('999')
location['location'] = location['location'].fillna('999')

In [20]:
# remake the locationunique column because there are duplicates as it seems like the subdistrict wasn't included correctly
location['locationunique'] = location['location'] + location['aslprojecttype'] + location['district'] + location['sub_district']

In [23]:
print(f"There are now ",len(location[location['locationunique'].isnull()]), " are NAs in the locationunique column.")

There are now  0  are NAs in the locationunique column.


# Gear data

In [ ]:
# view the gaer data
gear.head()

,Gear,SASAP.Gear
0,NaN,NaN
1,handpicked or carcass,handpicked or carcass
2,beach seine,seine
3,sport hook and line,sport hook and line
4,weir,weir


In [14]:
# assess if the gear data is in the right data fromats
gear.dtypes

Gear          object
SASAP.Gear    object
dtype: object

In [ ]:

location.head()

,LocationUnique,SASAP.Region_Corrected,Location,ASLProjectType,District,Sub.District,Lat,Lon,LocationID
0,Aleutians-commercial catch-302,Alaska Peninsula and Aleutian Islands,NaN,commercial catch,302,19,NaN,NaN,Aleutians
1,Bear River-commercial catch-31511,Alaska Peninsula and Aleutian Islands,Bear River,commercial catch,315,11,NaN,NaN,Bear River
2,Bear River-escapement-31511,Alaska Peninsula and Aleutian Islands,Bear River,escapement,315,11,56.0389,-160.2734,Bear River
3,Bear River-test fishing-31511,Alaska Peninsula and Aleutian Islands,Bear River,test fishing,315,11,56.0389,-160.2734,Bear River
4,Bear River-test fishing-31511,Alaska Peninsula and Aleutian Islands,Bear River,test fishing,316,11,56.0389,-160.2734,Bear River


In [77]:
# join the salmon data with the location data to get the unique location column
salmon_data = pd.merge(salmon_data, location, how = "inner", on = 'LocationID')

KeyError: 'LocationID'

In [52]:
salmon_data.head()

,Species,Length.Measurement.Type,sampleYear,LocationID,sampleDate,Length,Weight,Sex,Salt.Water.Age,Fresh.Water.Age,...,Stat.area,DataSource,LocationUnique,SASAP.Region_Corrected,Location,ASLProjectType,District,Sub.District,Lat,Lon
0,chinook,length not taken,1992,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,3.0,1.0,...,10510.0,ADFG Southeast and Westward,Affleck Canal/Spanish Is/Louise Cove-test fish...,Southeast,District 105 Test Seine,test fishing,105,10,NaN,NaN
1,chinook,length not taken,1992,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,3.0,1.0,...,10510.0,ADFG Southeast and Westward,Affleck Canal/Spanish Is/Louise Cove-test fish...,Southeast,District 105 Test Troll,test fishing,105,10,NaN,NaN
2,chinook,length not taken,1992,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,3.0,1.0,...,10510.0,ADFG Southeast and Westward,Affleck Canal/Spanish Is/Louise Cove-commercia...,Southeast,District 105 Traditional Seine,commercial catch,105,10,NaN,NaN
3,chinook,length not taken,1992,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,3.0,1.0,...,10510.0,ADFG Southeast and Westward,Affleck Canal/Spanish Is/Louise Cove-commercia...,Southeast,District 105 Traditional Troll,commercial catch,105,10,NaN,NaN
4,chinook,length not taken,1992,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,3.0,1.0,...,10510.0,ADFG Southeast and Westward,Affleck Canal/Spanish Is/Louise Cove-test fish...,Southeast,District 105 Troll Sublegal,test fishing,105,10,NaN,NaN


In [53]:
# reselect the columns we need
salmon_data = salmon_data[['Species', 'Length.Measurement.Type', 'sampleYear', 'sampleDate', 'Length', 'Weight', 'Sex','Salt.Water.Age', 'Fresh.Water.Age', 'Gear', 'Stat.area', 'DataSource', 'LocationUnique']]

# clean column names
salmon_data = salmon_data.clean_names()

In [54]:
salmon_data.head()

,species,length_measurement_type,sampleyear,sampledate,length,weight,sex,salt_water_age,fresh_water_age,gear,stat_area,datasource,locationunique
0,chinook,length not taken,1992,1992-03-31,NaN,NaN,examined but did not identify,3.0,1.0,troll,10510.0,ADFG Southeast and Westward,Affleck Canal/Spanish Is/Louise Cove-test fish...
1,chinook,length not taken,1992,1992-03-31,NaN,NaN,examined but did not identify,3.0,1.0,troll,10510.0,ADFG Southeast and Westward,Affleck Canal/Spanish Is/Louise Cove-test fish...
2,chinook,length not taken,1992,1992-03-31,NaN,NaN,examined but did not identify,3.0,1.0,troll,10510.0,ADFG Southeast and Westward,Affleck Canal/Spanish Is/Louise Cove-commercia...
3,chinook,length not taken,1992,1992-03-31,NaN,NaN,examined but did not identify,3.0,1.0,troll,10510.0,ADFG Southeast and Westward,Affleck Canal/Spanish Is/Louise Cove-commercia...
4,chinook,length not taken,1992,1992-03-31,NaN,NaN,examined but did not identify,3.0,1.0,troll,10510.0,ADFG Southeast and Westward,Affleck Canal/Spanish Is/Louise Cove-test fish...


In [55]:
# look at the unique values
print(salmon_data['species'].unique())
print(salmon_data['length_measurement_type'].unique())
print(salmon_data['sampleyear'].unique())
print(salmon_data['sampledate'].unique())
print(salmon_data['length'].unique())
print(salmon_data['weight'].unique())
print(salmon_data['sex'].unique())
print(salmon_data['salt_water_age'].unique())
print(salmon_data['fresh_water_age'].unique())
print(salmon_data['gear'].unique())
print(salmon_data['stat_area'].unique())
print(salmon_data['locationunique'].unique())

['chinook' 'chum' 'coho' 'pink' 'sockeye']
['length not taken' 'mid-eye to fork of tail' 'mid-eye to hypural plate'
 'post orbit to hypural plate' 'post-orbit to fork of tail'
 'tip of snout to fork of tail' 'tip of snout to tip of tail' 'unknown'
 nan 'cleithral arch to fork']
<IntegerArray>
[1992, 2002, 1997, 1999, 2000, 2001, 2003, 1996, 1991, 1995, 2007, 1998, 1969,
 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, <NA>, 1979, 1980, 1981,
 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1993, 1994, 2004, 2005,
 2006, 2008, 2009, 2010, 2011, 2012, 1966, 1968, 2013, 2014, 2015, 2016, 1964,
 1965, 1967, 2017, 1961, 1959, 1962, 1963, 1957, 1960, 2018, 1937, 1958]
Length: 64, dtype: Int64
['1992-03-31' '2002-07-15' '2002-08-30' ... '1977-05-26' '1977-06-02'
 '1978-05-20']
[  nan  781.  768. ...  114.  113. 5440.]
[  nan  498.  320. ... 2080.  654. 2780.]
['examined but did not identify' 'male' 'female' 'unknown' nan '2' '1' 1.0
 2.0 0.0 3.0 '3']
[ 3.  4. nan  2.  1.  5.  6.  

In [56]:
# change weird unknowns to NAs
salmon_data.loc[salmon_data['length_measurement_type']== 'unknown', 'length_measurement_type'] = np.nan
print(salmon_data['length_measurement_type'].unique())

['length not taken' 'mid-eye to fork of tail' 'mid-eye to hypural plate'
 'post orbit to hypural plate' 'post-orbit to fork of tail'
 'tip of snout to fork of tail' 'tip of snout to tip of tail' nan
 'cleithral arch to fork']


In [57]:
# change weird unknowns to NA
# filter rows that are not 'male' and not 'female'
invalid_mask = (~salmon_data['sex'].isin(['male', 'female']))

# fill these rows with NA
salmon_data.loc[invalid_mask, 'sex'] = np.nan

print(salmon_data['sex'].unique())

[nan 'male' 'female']


In [ ]:
# save data to a csv file
# salmon_data.to_csv('salmon_data_clean.csv')

## Clean location dataframe

In [62]:
print(location['locationunique'].unique())
print(location['sasap_region_corrected'].unique())
print(location['location'].unique())
print(location['aslprojecttype'].unique())
print(location['district'].unique())
print(location['sub_district'].unique())
print(location['lat'].unique())
print(location['lon'].unique())
print(location['locationid'].unique())

['Aleutians-commercial catch-302' 'Bear River-commercial catch-31511'
 'Bear River-escapement-31511' ...
 'Yukon Crossing, Y.T., Canada (Village/City)-escapement-NA'
 'Yukon District-subsistence catch-NA' nan]
['Alaska Peninsula and Aleutian Islands' 'Bristol Bay' 'Chignik'
 'Cook Inlet' 'Copper River' 'Kodiak' 'Kotzebue' 'Kuskokwim'
 'Norton Sound' 'Prince William Sound' 'Southeast' 'Yukon' nan]
[nan 'Bear River' 'Bear River (genetics)' ... 'Y6 (Subdistrict 6)'
 'Yukon Crossing, Y.T., Canada (Village/City)' 'Yukon District']
['commercial catch' 'escapement' 'test fishing' nan 'sport catch'
 'hatchery cost recovery' 'mark/recapture' 'personal use' 'brood stock'
 'subsistence catch' 'test fish' 'unknown' 'commercial common property'
 'mixed ccp and hcr' 'sport fish' 'brood excess']
['302' '315' '316' '311' '312' '284' '283' '324' '313' '282' '285' nan
 '318' '314' '31' '317' '303' '281' '322' '325' '326' '321' '271' '272'
 '275' '273' '270' '259' '231' '241' '244' '249' '247' '240' '245

In [73]:
# view duplicates based on the location unique
dup2_location = location[location.duplicated(subset = ['locationunique'], keep = False)]
dup2_location

,locationunique,sasap_region_corrected,location,aslprojecttype,district,sub_district,lat,lon,locationid
337,Chignik Lagooncommercial catch27110,Chignik,Chignik Lagoon,commercial catch,271,10,NaN,NaN,Chignik Lagoon
338,Chignik Lagooncommercial catch27110,Chignik,Chignik Lagoon,commercial catch,271,10,NaN,NaN,Chignik Lagoon
479,Bering Rivercommercial catch2000,Copper River,Bering River,commercial catch,200,0,NaN,NaN,Bering River
480,Bering Rivercommercial catch2000,Copper River,Bering River,commercial catch,200,0,NaN,NaN,Bering River
485,Copper Rivercommercial catch2120,Copper River,Copper River,commercial catch,212,0,NaN,NaN,Copper River
488,Copper Rivercommercial catch2120,Copper River,Copper River,commercial catch,212,0,NaN,NaN,Copper River
1340,Prince William Sound Generalcommercial catch2200,Prince William Sound,Prince William Sound General,commercial catch,220,0,NaN,NaN,Prince William Sound
1342,Prince William Sound Generalcommercial catch2200,Prince William Sound,Prince William Sound General,commercial catch,220,0,NaN,NaN,Prince William Sound General


In [74]:
# drop duplicates in unique location 
location = location.drop_duplicates(subset = 'locationunique', keep = 'first')

In [ ]:
# save data to a csv file
# location.to_csv('location_clean.csv')

## Clean length_type

In [ ]:
length_type = length_type.clean_names()

In [ ]:
length_type

,length_measurement_type,length_fill,locationid,species,aslprojecttype,sampleyear
0,mid-eye to fork of tail,NaN,108 Creek,coho,escapement,1998
1,mid-eye to fork of tail,NaN,108 Creek,coho,escapement,1999
2,mid-eye to fork of tail,NaN,108 Creek,coho,escapement,2000
3,mid-eye to fork of tail,NaN,18 Mile Slough,chum,escapement,2003
4,mid-eye to fork of tail,NaN,18 Mile Slough,chum,escapement,2004
...,...,...,...,...,...,...
16522,mid-eye to fork of tail,NaN,Zimovia Straight,chinook,commercial catch,1991
16523,mid-eye to fork of tail,NaN,Zimovia Straight,chinook,commercial catch,2004
16524,mid-eye to fork of tail,NaN,Zimovia Straight,chinook,commercial catch,2005
16525,mid-eye to fork of tail,NaN,Zimovia Straight,chinook,commercial catch,2007


In [ ]:
print(length_type['length_measurement_type'].unique())
print(length_type['length_fill'].unique())
print(length_type['locationid'].unique)

['mid-eye to fork of tail' 'length not taken' nan
 'post orbit to fork of tail' 'tip of snout to fork of tail'
 'post orbit to hypural plate' 'mid-eye to hypural plate'
 'tip of snout to tip of tail' 'post-orbit to fork of tail'
 'cleithral arch to fork']
[nan 'unknown' 'mid-eye to fork of tail']
<bound method Series.unique of 0               108 Creek
1               108 Creek
2               108 Creek
3          18 Mile Slough
4          18 Mile Slough
               ...       
16522    Zimovia Straight
16523    Zimovia Straight
16524    Zimovia Straight
16525    Zimovia Straight
16526    Zimovia Straight
Name: locationid, Length: 16527, dtype: object>


In [ ]:
length_type.loc[length_type['length_measurement_type']== 'unknown', 'length_measurement_type'] = np.nan
print(length_type['length_measurement_type'].unique())

['mid-eye to fork of tail' 'length not taken' nan
 'post orbit to fork of tail' 'tip of snout to fork of tail'
 'post orbit to hypural plate' 'mid-eye to hypural plate'
 'tip of snout to tip of tail' 'post-orbit to fork of tail'
 'cleithral arch to fork']


## Clean gear dataframe

In [ ]:
gear.head()

,Gear,SASAP.Gear
0,NaN,NaN
1,handpicked or carcass,handpicked or carcass
2,beach seine,seine
3,sport hook and line,sport hook and line
4,weir,weir


In [ ]:
# clean names 
gear = gear.clean_names()

gear.dropna(inplace=True)

In [ ]:
gear.head()

,gear,sasap_gear
1,handpicked or carcass,handpicked or carcass
2,beach seine,seine
3,sport hook and line,sport hook and line
4,weir,weir
5,drift gillnet,gillnet


In [ ]:
# save data to a csv file
# gear.to_csv('gear_clean.csv')